# MVCC
Multi-Version Concurrency Control
多版本并发控制

PostgreSQL 通过保存多个行版本，让不同事务根据自己的快照看到合适的数据版本，从而减少读写之间的互相阻塞。

### 为什么 PostgreSQL 的 UPDATE 更像“创建新版本”，而不是直接覆盖旧值。
主要为了避免并发状态下，避免读取并发事务产生的不一致状态。

UPDATE 创建新版本，旧版本暂时保留给仍可能需要它的事务。

比如：初始为100，事务1开始查询，基于当前数据进行一份长时间统计，这时候2把100变成50，如果直接覆盖，原先1看到的稳定数据就消失了。
所以就有了MVCC，1和2各自基于当前快照，1继续统计100，后续的事务的快照就是50。


### MVCC的主要并发优势
MVCC 使普通读取与写入在很多情况下不需要互相等待。

旧事务继续读旧版本，新事务写入新版本。

但是仍然可能发生冲突：
- 2个事务同时更新同一行
- select ... for update 可能阻塞写入
- 修改表结构
- 唯一约束竞争
- 显式表锁或行锁

### 多版本的代价
可能积累大量已经不再需要的旧版本。

### 记忆模型
可以理解成是一个在线文档系统，修改不会立刻销毁旧稿。不同人进来根据进入文档的时间和权限看到对应版本。

## 快照可见性
### Read Committed
pgsql 默认隔离级别
```sql
事务 A 开始
    SELECT balance;  → 100

事务 B
    UPDATE balance = 80;
    COMMIT;

事务 A
    SELECT balance;  → 80
```
Read Committed 下，通常每条 SQL 语句开始时都会取得新的快照。

### Repeatable Read
```sql
事务 A 开始
    SELECT balance;  → 100

事务 B
    UPDATE balance = 80;
    COMMIT;

事务 A
    SELECT balance;  → 100
```
只有事务 A 结束，再开启新事务，才会看到 80

### 事务通常可以看到自己尚未提交的修改
```sql
BEGIN;

UPDATE accounts
SET balance = 80
WHERE id = 1;

SELECT balance
FROM accounts
WHERE id = 1;


-- 其他事务未提交的修改 → 看不到
-- 当前事务自己的修改   → 看得到
-- 先过滤未提交，再根据快照选版本；自己的修改自己看得见。
```

### 死元祖与VACUUM
##### 学习目标
- 理解旧行版本如何变成“垃圾”，以及 PostgreSQL 为什么需要定期清理它们。
##### 本节重点
- 什么是 dead tuple（死元组）
- 普通 VACUUM 做什么
- VACUUM FULL 和普通 VACUUM 的区别
- autovacuum 为什么不能随便关闭

### 什么是死元祖？
死元组已经不能被正常查询看到，但它仍可能占用表和索引中的空间。

也就是旧版本对所有事务都不再可见。

比如一次update开始：
```sql
-- (balance = 100) <-旧版本
UPDATE accounts
SET balance = 80
WHERE id = 1;
```

### 普通 VACUUM 做什么？
把表想象成停车场，旧车报废后，普通VACUUM会把车位标记为可以重新使用。以后插入或更新的数据可以重复利用这些空间。

> 空间可被数据库复用!=立即归还给操作系统

所以执行普通VACUUM后：
- 死元祖占用空间可以被后续数据复用
- 表文件大小通常不会立刻明显缩小
- 通常可以与正常查询和增删改并行工作

### VACUUM FULL 做什么？
VACUUM FULL 会彻底删除死元组，并回收空间。
但代价很大：
- 需要重写整张表
- 速度更慢
- 需要额外磁盘空间
- 会取得access exclusive锁，阻塞其他对该表的访问。

通常只在严重表膨胀等特定场景考虑。

### autovacuum
PostgreSQL 通常由后台的 autovacuum 自动判断哪些表需要清理

它的工作不只是删除垃圾，还包括：
- 回收死元组空间供数据库复用；
- 更新查询优化器使用的统计信息；
- 维护可见性信息；
- 防止事务 ID 回卷问题。

### 为什么不能随便关闭 autovacuum？
autovacuum 是 PostgreSQL 自动管理的重要组成部分，关闭它可能导致：
- 表膨胀失控
- 查询性能下降
- 空间浪费

**UPDATE 留旧版，旧版没人看才死亡；VACUUM 腾车位，FULL 才重建停车场。**

## 长事务与表膨胀
案例：
```
10:00 事务 A 建立快照，看到 balance = 100
之后一直没有提交

10:01 事务 B：100 → 90
10:02 事务 C： 90 → 80
10:03 事务 D： 80 → 70

现在存在多个版本：
100 → 90 → 80 → 70
 ↑                 ↑
A 可能仍要看       最新版本

即使 90、80 已经不再是最新数据，数据库也不能只根据“它不是最新版本”就清理。

事务 A 长时间不结束，就可能拖住 VACUUM 的清理边界。
```

#### 因果链
长事务可能引发或加重表膨胀的原因。
```
长事务持有旧快照
        ↓
旧版本暂时不能安全移除
        ↓
死元组持续堆积
        ↓
表和索引占用更多页面
        ↓
扫描更多数据、缓存利用率下降
        ↓
查询和维护成本增加
```

长事务握着旧照片不放，VACUUM 就不敢扔掉旧底片。